In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

from topostats.io import LoadScans
from topostats.plottingfuncs import Colormap

CMAP = Colormap().get_cmap()

import skimage.filters

VMIN = -3
VMAX = 4

In [ ]:
def adapted_sigma(
    feature_size_nm: float,
    pixel_to_nm_scaling: float,
) -> float:
    """Adjust the sigma to real scales for consistency."""
    return feature_size_nm / (2 * np.sqrt(2) * pixel_to_nm_scaling)


def change_image_scale(
    image: npt.NDArray, pixel_to_nm_scaling: float, scale_multiplier: float
) -> tuple[npt.NDArray, float]:
    """Change the resolution of the image and pixel to nm scaling factor"""
    new_pixel_to_nm_scaling = pixel_to_nm_scaling / scale_multiplier
    new_image = skimage.transform.rescale(
        image,
        scale=scale_multiplier,
        anti_aliasing=True,
        preserve_range=True,
    )
    return new_image, new_pixel_to_nm_scaling


file = Path("/Users/sylvi/topo_data/minicircle/output/data/processed/minicircle.topostats")
assert file.exists()

loadscans = LoadScans(img_paths=[file], config={"loading": {"channel": "dummy"}})
loadscans.get_data()

img_dict = loadscans.img_dict

image_data = img_dict["minicircle.topostats"]
pixel_to_nm_scaling = image_data.pixel_to_nm_scaling
assert pixel_to_nm_scaling is not None
pixel_to_nm_scaling = float(pixel_to_nm_scaling)

image = image_data.image
image = image
assert image is not None
image_size = image.shape[0]
image = image[int(image_size / 2) :, int(image_size / 2) :]

plt.imshow(image, vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.show()

# for sigma_strength in [1, 2, 3, 4, 5]:
#     image = skimage.filters.hessian(image, sigmas=[sigma_strength])
#     plt.imshow(image, vmin=VMIN, vmax=VMAX, cmap=CMAP)
#     plt.title(f"Hessian filter with sigma={sigma_strength}")
#     plt.show()


# for scale_range in [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]:
#     image = skimage.filters.hessian(image, scale_range=scale_range)
#     plt.imshow(image, vmin=VMIN, vmax=VMAX, cmap=CMAP)
#     plt.title(f"Hessian filter with scale_range={scale_range}")
#     plt.show()

# scale_range = (1, 2, 10)
# image = skimage.filters.hessian(image, scale_range=scale_range)
# plt.imshow(image, vmin=VMIN, vmax=VMAX, cmap=CMAP)
# plt.title(f"Hessian filter with scale_range={scale_range}")
# plt.show()

# scale_range = (1, 2, 3)
# image = skimage.filters.hessian(image, scale_range=scale_range)
# plt.imshow(image, vmin=VMIN, vmax=VMAX, cmap=CMAP)
# plt.title(f"Hessian filter with scale_range={scale_range}")
# plt.show()

feature_size_nm = 2.0
hessian_threshold = 0.8
sigma_strength = adapted_sigma(
    feature_size_nm=feature_size_nm,
    pixel_to_nm_scaling=pixel_to_nm_scaling,
)
beta = 0.1

image_hess = skimage.filters.hessian(image, sigmas=[sigma_strength])
plt.imshow(image_hess, vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.title(f"Hessian filter with sigma={sigma_strength:.2f}, beta: {beta}\n p2nm: {pixel_to_nm_scaling}")
plt.colorbar()
print(np.min(image_hess), np.max(image_hess))
plt.show()

image_hess_thresholded = image_hess > hessian_threshold
plt.imshow(image_hess)
plt.show()

image_lower_resolution, pixel_to_nm_scaling_lower_resolution = change_image_scale(
    image=image,
    pixel_to_nm_scaling=pixel_to_nm_scaling,
    scale_multiplier=0.5,
)

sigma_strength_low_res = adapted_sigma(
    feature_size_nm=feature_size_nm,
    pixel_to_nm_scaling=pixel_to_nm_scaling_lower_resolution,
)
plt.imshow(image_lower_resolution, vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.show()
image_lower_resolution_hess = skimage.filters.hessian(image_lower_resolution, sigmas=[sigma_strength_low_res])
plt.imshow(image_lower_resolution_hess, vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.title(
    f"Hessian filter with sigma={sigma_strength_low_res:.2f}, beta: {beta}\n p2nm: {pixel_to_nm_scaling_lower_resolution}"
)
plt.colorbar()
print(np.min(image_lower_resolution_hess), np.max(image_lower_resolution_hess))
plt.show()